# SUMO-Only Local ACO (Step-by-Step)

This notebook is a clean walkthrough of the full workflow:

1. Load SUMO graph and static edge weights
2. Explain weighting terms and visualize edge priorities
3. Define drone/ACO decision mechanics
4. Run simulation with readable state updates
5. Produce analysis and all visual diagnostics (same illustration set)

It keeps only the **main connected component** of the road graph to avoid skewing stats with unreachable subgraphs.

## 1) Imports

In [ ]:
import csv
import math
import random
from io import BytesIO

import numpy as np
import networkx as nx

try:
    import matplotlib.pyplot as plt
    import matplotlib.colors as mcolors
    HAS_MPL = True
except Exception:
    HAS_MPL = False
    print('matplotlib not available; plotting cells will be skipped.')

try:
    import imageio.v2 as imageio
    HAS_IMAGEIO = True
except Exception:
    HAS_IMAGEIO = False

try:
    from IPython.display import display, Image
    HAS_IPY = True
except Exception:
    HAS_IPY = False

from metrics import load_sumo_network, build_graphs

## 2) Load SUMO graph + static edge weights

Weights come from `sumo_only_edge_weights.csv` with columns:
- `f_demand`
- `f_topo`
- `f_redundancy`
- combined `W`

Combined static priority used for patrol:

`W = 0.5*f_demand + 0.3*f_topo + 0.2*f_redundancy`

In [ ]:
NET_FILE = '../../../../../../Examples/NFDRS4_Behave/Roxborough/sumo/rox_big.net.xml'
WEIGHTS_CSV = 'sumo_only_edge_weights.csv'

sumo_net = load_sumo_network(NET_FILE)
DG, UG_raw = build_graphs(sumo_net)

# Keep only largest connected component (main patrol subgraph).
components = sorted(nx.connected_components(UG_raw), key=len, reverse=True)
main_nodes = components[0]
UG = UG_raw.subgraph(main_nodes).copy()

print(f'Connected components found: {len(components)}')
print(f'Main component: {UG.number_of_nodes()} nodes, {UG.number_of_edges()} edges')
print(f'Disabled subgraphs: {UG_raw.number_of_nodes()-UG.number_of_nodes()} nodes, {UG_raw.number_of_edges()-UG.number_of_edges()} edges')

def as_float(v, default=0.0):
    try:
        if v is None or v == '':
            return default
        x = float(v)
        if math.isnan(x) or math.isinf(x):
            return default
        return x
    except Exception:
        return default

edge_metrics = {
    e: {'W': 0.0, 'f_demand': 0.0, 'f_topo': 0.0, 'f_redundancy': 0.0} 
    for e in UG.edges()
}

with open(WEIGHTS_CSV) as f:
    for row in csv.DictReader(f):
        u, v = row['u'], row['v']
        if UG.has_edge(u, v):
            e = (u, v)
        elif UG.has_edge(v, u):
            e = (v, u)
        else:
            continue

        edge_metrics[e]['W'] = as_float(row.get('W'), 0.0)
        edge_metrics[e]['f_demand'] = as_float(row.get('f_demand'), 0.0)
        edge_metrics[e]['f_topo'] = as_float(row.get('f_topo'), 0.0)
        edge_metrics[e]['f_redundancy'] = as_float(row.get('f_redundancy'), 0.0)

W = {e: edge_metrics[e]['W'] for e in UG.edges()}
F_DEMAND = {e: edge_metrics[e]['f_demand'] for e in UG.edges()}
F_TOPO = {e: edge_metrics[e]['f_topo'] for e in UG.edges()}
F_RED = {e: edge_metrics[e]['f_redundancy'] for e in UG.edges()}

w_arr = np.array(list(W.values()), dtype=float)
print(f'Loaded weighted edges: {len(W)}')
print(f'W mean/p95/max: {w_arr.mean():.4f} / {np.percentile(w_arr,95):.4f} / {w_arr.max():.4f}')

In [ ]:
print('{:<35} {:>7} {:>8} {:>8} {:>7}'.format('edge', 'W', 'f_dem', 'f_topo', 'f_red'))
print('-' * 75)
for (u, v), w in sorted(W.items(), key=lambda kv: -kv[1])[:20]:
    print('{:<15}-> {:<15} {:>7.3f} {:>8.3f} {:>8.3f} {:>7.3f}'.format(str(u)[:15], str(v)[:15], w, F_DEMAND[(u,v)], F_TOPO[(u,v)], F_RED[(u,v)]))

In [ ]:
def draw_edge_metric_map(G, values, title, cmap='viridis', vmin=None, vmax=None, robust_p95=False, log_scale=False):
    if not HAS_MPL:
        print('Skipping map: matplotlib not available.')
        return

    pos = {n: (G.nodes[n]['x'], G.nodes[n]['y']) for n in G.nodes()}
    edgelist = list(G.edges())
    vals = np.array([values[e] for e in edgelist], dtype=float)
    vals_draw = np.log1p(vals) if log_scale else vals

    if robust_p95 and vmax is None:
        vmax = float(np.percentile(vals_draw, 95))
    if vmin is None:
        vmin = float(vals_draw.min())
    if vmax is None:
        vmax = float(vals_draw.max())
    if vmax <= vmin:
        vmax = vmin + 1e-9

    norm = mcolors.Normalize(vmin=vmin, vmax=vmax)
    colors = plt.get_cmap(cmap)(norm(vals_draw))
    widths = 0.2 + 1.8 * np.clip((vals_draw - vmin) / (vmax - vmin), 0, 1)

    fig, ax = plt.subplots(figsize=(10, 9))
    nx.draw_networkx_edges(G, pos, edgelist=edgelist, edge_color=colors, width=widths, ax=ax)
    sm = plt.cm.ScalarMappable(cmap=plt.get_cmap(cmap), norm=norm)
    sm.set_array([])
    plt.colorbar(sm, ax=ax, shrink=0.75, label=title)
    ax.set_title(title)
    ax.set_aspect('equal')
    ax.axis('off')
    plt.tight_layout()
    plt.show()

if HAS_MPL:
    fig, axes = plt.subplots(2, 2, figsize=(18, 16))
    pos = {n: (UG.nodes[n]['x'], UG.nodes[n]['y']) for n in UG.nodes()}
    edgelist = list(UG.edges())

    def draw_on_ax(ax, values, title, cmap='viridis', robust_p95=False):
        vals = np.array([values[e] for e in edgelist], dtype=float)
        vmin = float(vals.min())
        vmax = float(np.percentile(vals, 95)) if robust_p95 else float(vals.max())
        if vmax <= vmin:
            vmax = vmin + 1e-9
        norm = mcolors.Normalize(vmin=vmin, vmax=vmax)
        colors = plt.get_cmap(cmap)(norm(vals))
        widths = 0.2 + 1.8 * np.clip((vals - vmin) / (vmax - vmin), 0, 1)
        nx.draw_networkx_edges(UG, pos, edgelist=edgelist, edge_color=colors, width=widths, ax=ax)
        sm = plt.cm.ScalarMappable(cmap=plt.get_cmap(cmap), norm=norm)
        sm.set_array([])
        plt.colorbar(sm, ax=ax, shrink=0.72)
        ax.set_title(title)
        ax.set_aspect('equal')
        ax.axis('off')

    draw_on_ax(axes[0,0], F_DEMAND, 'f_demand', cmap='YlOrRd')
    draw_on_ax(axes[0,1], F_TOPO, 'f_topo', cmap='viridis', robust_p95=True)
    draw_on_ax(axes[1,0], F_RED, 'f_redundancy', cmap='Reds')
    draw_on_ax(axes[1,1], W, 'Combined W (expected attention)', cmap='plasma', robust_p95=True)
    plt.tight_layout()
    plt.show()
else:
    print('Skipping weighting maps because matplotlib is not available.')

## 3) Configuration (fleet + ACO + runtime)

In [ ]:
# Fleet parameters
DroneCount = 20
CruiseSpeedMps = 75.0 / 3.6
BatteryEnduranceSeconds = 270000.0   # high battery for near-unconstrained patrol behavior
ReturnReserveFraction = 0.10
RechargeDurationSeconds = 1200.0
AutoPlaceChargingStationBottomLeft = True
ChargingStationSimulationPos = (0.0, 0.0)
UniformTypeId = 'default'
SPAWN_MODE = 'station'  # 'station' or 'random'

# Simulation
RNG_SEED = 42
SIM_HORIZON_S = 36000
DT_S = 1.0
METRIC_EVERY_S = 30

# ACO
ALPHA = 0.5
BETA = 2.0
GAMMA_OVERLAP = 1.5
AGE_SCALE_S = 900.0
PHEROMONE_INIT = 1.0
LOCAL_DECAY = 0.02
DEPOSIT_GAIN = 0.15
GLOBAL_EVAP_RATE = 0.0005

# Derived
NUM_DRONES = DroneCount
CRUISE_SPEED_MPS = CruiseSpeedMps
BATTERY_ENDURANCE_S = BatteryEnduranceSeconds
RESERVE_SECONDS = BatteryEnduranceSeconds * ReturnReserveFraction
RECHARGE_DURATION_S = RechargeDurationSeconds

print(f'Drones={NUM_DRONES}, speed={CRUISE_SPEED_MPS:.2f} m/s ({CRUISE_SPEED_MPS*3.6:.1f} km/h), spawn={SPAWN_MODE}')
print(f'Battery={BATTERY_ENDURANCE_S:.0f}s, reserve={RESERVE_SECONDS:.0f}s, recharge={RECHARGE_DURATION_S:.0f}s')

## 4) Agent decision mechanics

At each local decision, drone selects among adjacent edges that are battery-safe:

`score(e) = tau(e)^ALPHA * heuristic(e)^BETA / (1 + inflight(e))^GAMMA_OVERLAP`

with:
- `heuristic(e) = 0.4*W(e) + 0.6*age_term(e)`
- `age_term = min(age/AGE_SCALE_S, 3)`
- local pheromone update on selection
- global evaporation every simulation step

In [ ]:
def run_simulation(UG, W, settings):
    rng = random.Random(settings['RNG_SEED'])

    edges = list(UG.edges())
    nodes = [n for n in UG.nodes() if UG.degree(n) > 0]
    edge_length = {(u, v): max(float(UG[u][v].get('length', 1.0)), 1.0) for (u, v) in edges}

    coords = {n: (float(UG.nodes[n]['x']), float(UG.nodes[n]['y'])) for n in nodes}
    if settings['AutoPlaceChargingStationBottomLeft']:
        target_x = min(x for x, _ in coords.values())
        target_y = min(y for _, y in coords.values())
    else:
        target_x, target_y = settings['ChargingStationSimulationPos']

    station_node = min(nodes, key=lambda n: (coords[n][0] - target_x) ** 2 + (coords[n][1] - target_y) ** 2)
    station_xy = coords[station_node]

    station_dist_m = nx.single_source_dijkstra_path_length(UG, source=station_node, weight='length')
    time_to_station = {n: station_dist_m.get(n, np.inf) / settings['CRUISE_SPEED_MPS'] for n in nodes}

    if settings['SPAWN_MODE'] == 'random':
        start_nodes = [rng.choice(nodes) for _ in range(settings['NUM_DRONES'])]
    else:
        start_nodes = [station_node for _ in range(settings['NUM_DRONES'])]

    drones = []
    for i in range(settings['NUM_DRONES']):
        drones.append({
            'id': i,
            'node': start_nodes[i],
            'edge': None,
            'target': None,
            'remaining': 0.0,
            'state': 'patrol',
            'battery_s': settings['BATTERY_ENDURANCE_S'],
            'charge_remaining_s': 0.0,
            'recharge_count': 0,
            'distance_m': 0.0,
            'patrol_scans': 0,
        })

    tau = {e: settings['PHEROMONE_INIT'] for e in edges}
    last_visit = {e: 0.0 for e in edges}
    inflight = {e: 0 for e in edges}
    visit_counts = {e: 0 for e in edges}
    visit_times = {e: [] for e in edges}

    def edge_key(u, v):
        return (u, v) if (u, v) in tau else (v, u)

    def edge_travel_time_s(u, v):
        e = edge_key(u, v)
        return max(edge_length[e] / settings['CRUISE_SPEED_MPS'], 1.0)

    def heuristic_value(e, t):
        age = t - last_visit[e]
        age_term = min(age / settings['AGE_SCALE_S'], 3.0)
        return 1e-6 + 0.4 * W[e] + 0.6 * age_term

    def should_return_now(drone):
        tts = time_to_station.get(drone['node'], np.inf)
        if not np.isfinite(tts):
            return False
        return drone['battery_s'] <= (tts + settings['RESERVE_SECONDS'])

    def can_safely_traverse(u, v, battery_s):
        t_edge = edge_travel_time_s(u, v)
        tts_next = time_to_station.get(v, np.inf)
        if not np.isfinite(tts_next):
            return False
        return (battery_s - t_edge - tts_next) >= settings['RESERVE_SECONDS']

    def choose_next_edge(node, t, battery_s):
        nbrs = list(UG.neighbors(node))
        if not nbrs:
            return None, None

        scores = []
        for v in nbrs:
            if not can_safely_traverse(node, v, battery_s):
                continue
            e = edge_key(node, v)
            h = heuristic_value(e, t)
            s = (tau[e] ** settings['ALPHA']) * (h ** settings['BETA']) / ((1.0 + inflight[e]) ** settings['GAMMA_OVERLAP'])
            scores.append((v, e, max(s, 1e-12)))

        if not scores:
            return None, None

        total = sum(s for _, _, s in scores)
        r = rng.random() * total
        c = 0.0
        for v, e, s in scores:
            c += s
            if c >= r:
                return v, e
        return scores[-1][0], scores[-1][1]

    def start_edge_traversal(drone, target_node, edge, mode):
        inflight[edge] += 1
        t_edge = edge_travel_time_s(drone['node'], target_node)
        drone['edge'] = edge
        drone['target'] = target_node
        drone['remaining'] = t_edge
        drone['state'] = mode
        drone['distance_m'] += edge_length[edge]

    def enter_charge(drone):
        drone['state'] = 'charge'
        drone['edge'] = None
        drone['target'] = None
        drone['remaining'] = 0.0
        drone['charge_remaining_s'] = settings['RECHARGE_DURATION_S']
        drone['recharge_count'] += 1

    def dispatch_return(drone, t):
        if drone['node'] == station_node:
            enter_charge(drone)
            return
        path = nx.shortest_path(UG, source=drone['node'], target=station_node, weight='length')
        if len(path) < 2:
            enter_charge(drone)
            return
        nxt = path[1]
        e = edge_key(drone['node'], nxt)
        start_edge_traversal(drone, nxt, e, mode='return')

    def dispatch_patrol(drone, t):
        if should_return_now(drone):
            dispatch_return(drone, t)
            return

        nxt, e = choose_next_edge(drone['node'], t, drone['battery_s'])
        if e is None:
            dispatch_return(drone, t)
            return

        tau[e] = (1.0 - settings['LOCAL_DECAY']) * tau[e] + settings['LOCAL_DECAY'] * settings['PHEROMONE_INIT']
        start_edge_traversal(drone, nxt, e, mode='patrol')

    # Initial dispatch
    for d in drones:
        dispatch_patrol(d, 0.0)

    # History containers
    times = []
    mean_weighted_age = []
    max_weighted_age = []
    p95_weighted_age = []
    recent_coverage = []

    battery_min = []
    battery_mean = []
    battery_max = []
    state_patrol = []
    state_return = []
    state_charge = []

    # Main loop
    steps = int(settings['SIM_HORIZON_S'] / settings['DT_S'])
    for step in range(1, steps + 1):
        t = step * settings['DT_S']

        # Global evaporation
        for e in edges:
            tau[e] *= (1.0 - settings['GLOBAL_EVAP_RATE'])

        # Drone updates
        for d in drones:
            if d['state'] == 'charge':
                charge_rate = settings['BATTERY_ENDURANCE_S'] / settings['RECHARGE_DURATION_S']
                d['battery_s'] = min(settings['BATTERY_ENDURANCE_S'], d['battery_s'] + charge_rate * settings['DT_S'])
                d['charge_remaining_s'] -= settings['DT_S']
                if d['charge_remaining_s'] <= 1e-12:
                    d['battery_s'] = settings['BATTERY_ENDURANCE_S']
                    d['charge_remaining_s'] = 0.0
                    d['state'] = 'patrol'
                    dispatch_patrol(d, t)
                continue

            if d['edge'] is None:
                if d['state'] == 'return':
                    dispatch_return(d, t)
                else:
                    dispatch_patrol(d, t)
                continue

            d['remaining'] -= settings['DT_S']
            d['battery_s'] = max(0.0, d['battery_s'] - settings['DT_S'])

            while d['edge'] is not None and d['remaining'] <= 1e-12:
                e = d['edge']
                inflight[e] = max(inflight[e] - 1, 0)

                d['node'] = d['target']
                d['edge'] = None
                d['target'] = None
                d['remaining'] = 0.0

                if d['state'] == 'patrol':
                    age_before = t - last_visit[e]
                    last_visit[e] = t
                    visit_counts[e] += 1
                    d['patrol_scans'] += 1
                    visit_times[e].append(t)

                    age_term = min(age_before / settings['AGE_SCALE_S'], 1.0)
                    tau[e] += settings['DEPOSIT_GAIN'] * (0.5 * W[e] + 0.5 * age_term)
                    dispatch_patrol(d, t)
                elif d['state'] == 'return':
                    if d['node'] == station_node:
                        enter_charge(d)
                    else:
                        dispatch_return(d, t)

        # Metric snapshots
        if step % int(settings['METRIC_EVERY_S'] / settings['DT_S']) == 0:
            wa = np.array([W[e] * (t - last_visit[e]) for e in edges], dtype=float)
            times.append(t)
            mean_weighted_age.append(float(wa.mean()))
            max_weighted_age.append(float(wa.max()))
            p95_weighted_age.append(float(np.percentile(wa, 95)))
            recent_coverage.append(float(np.mean([(t - last_visit[e]) <= settings['AGE_SCALE_S'] for e in edges])))

            b = np.array([d['battery_s'] for d in drones], dtype=float)
            battery_min.append(float(b.min()))
            battery_mean.append(float(b.mean()))
            battery_max.append(float(b.max()))

            state_patrol.append(sum(1 for d in drones if d['state'] == 'patrol'))
            state_return.append(sum(1 for d in drones if d['state'] == 'return'))
            state_charge.append(sum(1 for d in drones if d['state'] == 'charge'))

    return {
        'edges': edges,
        'nodes': nodes,
        'station_node': station_node,
        'station_xy': station_xy,
        'time_to_station': time_to_station,
        'visit_counts': visit_counts,
        'visit_times': visit_times,
        'last_visit': last_visit,
        'drones': drones,
        'times': times,
        'mean_weighted_age': mean_weighted_age,
        'max_weighted_age': max_weighted_age,
        'p95_weighted_age': p95_weighted_age,
        'recent_coverage': recent_coverage,
        'battery_min': battery_min,
        'battery_mean': battery_mean,
        'battery_max': battery_max,
        'state_patrol': state_patrol,
        'state_return': state_return,
        'state_charge': state_charge,
    }

## 5) Run simulation

In [ ]:
settings = {
    'NUM_DRONES': NUM_DRONES,
    'CRUISE_SPEED_MPS': CRUISE_SPEED_MPS,
    'BATTERY_ENDURANCE_S': BATTERY_ENDURANCE_S,
    'RESERVE_SECONDS': RESERVE_SECONDS,
    'RECHARGE_DURATION_S': RECHARGE_DURATION_S,
    'AutoPlaceChargingStationBottomLeft': AutoPlaceChargingStationBottomLeft,
    'ChargingStationSimulationPos': ChargingStationSimulationPos,
    'SPAWN_MODE': SPAWN_MODE,
    'RNG_SEED': RNG_SEED,
    'SIM_HORIZON_S': SIM_HORIZON_S,
    'DT_S': DT_S,
    'METRIC_EVERY_S': METRIC_EVERY_S,
    'ALPHA': ALPHA,
    'BETA': BETA,
    'GAMMA_OVERLAP': GAMMA_OVERLAP,
    'AGE_SCALE_S': AGE_SCALE_S,
    'PHEROMONE_INIT': PHEROMONE_INIT,
    'LOCAL_DECAY': LOCAL_DECAY,
    'DEPOSIT_GAIN': DEPOSIT_GAIN,
    'GLOBAL_EVAP_RATE': GLOBAL_EVAP_RATE,
}

sim = run_simulation(UG, W, settings)

print(f"Simulation done: {SIM_HORIZON_S} s")
print(f"Spawn mode: {SPAWN_MODE}")
print(f"Final mean weighted age: {sim['mean_weighted_age'][-1]:.2f}")
print(f"Final p95 weighted age:  {sim['p95_weighted_age'][-1]:.2f}")
print(f"Final max weighted age:  {sim['max_weighted_age'][-1]:.2f}")
print(f"Final <= {AGE_SCALE_S:.0f}s freshness coverage: {sim['recent_coverage'][-1]*100:.1f}%")
print(f"Total recharge events: {sum(d['recharge_count'] for d in sim['drones'])}")

## 6) Primary outputs

In [ ]:
if HAS_MPL:
    fig, axes = plt.subplots(2, 2, figsize=(14, 9))

    axes[0, 0].plot(sim['times'], sim['mean_weighted_age'], label='mean weighted age')
    axes[0, 0].plot(sim['times'], sim['p95_weighted_age'], label='p95 weighted age')
    axes[0, 0].plot(sim['times'], sim['max_weighted_age'], label='max weighted age')
    axes[0, 0].set_xlabel('time [s]')
    axes[0, 0].set_ylabel('weighted age')
    axes[0, 0].set_title('Weighted age-of-information over time')
    axes[0, 0].legend()
    axes[0, 0].grid(alpha=0.3)

    axes[0, 1].plot(sim['times'], np.array(sim['recent_coverage']) * 100.0, color='tab:green')
    axes[0, 1].set_xlabel('time [s]')
    axes[0, 1].set_ylabel('coverage [%]')
    axes[0, 1].set_title(f'Edges with age <= {AGE_SCALE_S:.0f}s')
    axes[0, 1].grid(alpha=0.3)

    axes[1, 0].plot(sim['times'], sim['battery_min'], label='battery min')
    axes[1, 0].plot(sim['times'], sim['battery_mean'], label='battery mean')
    axes[1, 0].plot(sim['times'], sim['battery_max'], label='battery max')
    axes[1, 0].set_xlabel('time [s]')
    axes[1, 0].set_ylabel('battery [s]')
    axes[1, 0].set_title('Fleet battery profile')
    axes[1, 0].legend()
    axes[1, 0].grid(alpha=0.3)

    axes[1, 1].plot(sim['times'], sim['state_patrol'], label='patrol')
    axes[1, 1].plot(sim['times'], sim['state_return'], label='return')
    axes[1, 1].plot(sim['times'], sim['state_charge'], label='charge')
    axes[1, 1].set_xlabel('time [s]')
    axes[1, 1].set_ylabel('#drones')
    axes[1, 1].set_title('Drone state occupancy')
    axes[1, 1].legend()
    axes[1, 1].grid(alpha=0.3)

    plt.tight_layout()
    plt.show()
else:
    print('Skipping plots because matplotlib is not available.')

top = sorted(sim['edges'], key=lambda e: sim['visit_counts'][e], reverse=True)[:20]
print(f"{'edge':<35} {'visits':>8} {'rate/h':>8} {'W':>7} {'last_age_s':>11}")
print('-' * 84)
for u, v in top:
    age = SIM_HORIZON_S - sim['last_visit'][(u, v)]
    rate = sim['visit_counts'][(u, v)] / (SIM_HORIZON_S / 3600.0)
    print(f"{str(u)[:15]:<15}-> {str(v)[:15]:<15} {sim['visit_counts'][(u,v)]:>8d} {rate:>8.2f} {W[(u,v)]:>7.3f} {age:>11.1f}")

## 7) Scan-frequency analysis

In [ ]:
scan_counts = np.array([sim['visit_counts'][e] for e in sim['edges']], dtype=float)
w_vals = np.array([W[e] for e in sim['edges']], dtype=float)
last_ages = np.array([SIM_HORIZON_S - sim['last_visit'][e] for e in sim['edges']], dtype=float)
visited_mask = scan_counts > 0

visit_rate = np.zeros_like(scan_counts)
visit_rate[visited_mask] = scan_counts[visited_mask] / (SIM_HORIZON_S / 3600.0)

avg_revisit = np.full_like(scan_counts, np.nan)
avg_revisit[visited_mask] = SIM_HORIZON_S / scan_counts[visited_mask]

exact_intervals = []
for e in sim['edges']:
    vt = sim['visit_times'][e]
    if len(vt) >= 2:
        exact_intervals.extend(np.diff(vt))
exact_intervals = np.array(exact_intervals, dtype=float) if exact_intervals else np.array([], dtype=float)

print('Edge-scan summary')
print('------------------')
print(f'Total edge scans (all drones): {int(scan_counts.sum())}')
print(f'Simulation horizon:            {SIM_HORIZON_S:.0f} s ({SIM_HORIZON_S/3600.0:.2f} h)')
print(f'Aggregate scan rate:           {scan_counts.sum()/(SIM_HORIZON_S/3600.0):.1f} scans/h')
print(f'Per-drone scan rate:           {scan_counts.sum()/(SIM_HORIZON_S/3600.0)/NUM_DRONES:.2f} scans/h/drone')
print(f'Edges scanned at least once:   {int(visited_mask.sum())} / {len(sim["edges"])} ({100*visited_mask.mean():.1f}%)')
print(f'Edges never scanned:           {int((~visited_mask).sum())}')
print(f'Median scans per visited edge: {np.median(scan_counts[visited_mask]):.1f}')
print(f'P90 scans per visited edge:    {np.percentile(scan_counts[visited_mask], 90):.1f}')
print(f'Median avg revisit (visited):  {np.median(avg_revisit[visited_mask]):.1f} s')
print(f'P90 avg revisit (visited):     {np.percentile(avg_revisit[visited_mask], 90):.1f} s')
print(f'Median final edge age:         {np.median(last_ages):.1f} s')
print(f'P95 final edge age:            {np.percentile(last_ages, 95):.1f} s')
print(f'Edges fresher than {AGE_SCALE_S:.0f}s:    {100*np.mean(last_ages <= AGE_SCALE_S):.1f}%')
if exact_intervals.size > 0:
    print('Exact inter-visit intervals:')
    print(f'  Median / P90 / P95 [s]: {np.median(exact_intervals):.1f} / {np.percentile(exact_intervals,90):.1f} / {np.percentile(exact_intervals,95):.1f}')

## 8) Weight-stratified behavior

In [ ]:
q = np.quantile(w_vals, [0.0, 0.25, 0.50, 0.75, 1.0])
print('Scan behavior by weight quartile')
print('--------------------------------')
print(f"{'Weight bin':<22} {'#edges':>7} {'mean scans':>12} {'median scans':>13} {'median age[s]':>14} {'<=900s[%]':>10}")
for i in range(4):
    lo, hi = q[i], q[i+1]
    if i < 3:
        mask = (w_vals >= lo) & (w_vals < hi)
        label = f'Q{i+1} [{lo:.3f}, {hi:.3f})'
    else:
        mask = (w_vals >= lo) & (w_vals <= hi)
        label = f'Q{i+1} [{lo:.3f}, {hi:.3f}]'
    c = scan_counts[mask]
    a = last_ages[mask]
    print(f"{label:<22} {c.size:>7d} {c.mean():>12.2f} {np.median(c):>13.2f} {np.median(a):>14.1f} {100*np.mean(a <= AGE_SCALE_S):>10.1f}")

## 9) Scan distribution diagnostics

In [ ]:
if HAS_MPL:
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))

    axes[0,0].hist(scan_counts, bins=40, color='tab:blue', alpha=0.8)
    axes[0,0].set_title('Distribution of edge scan counts')
    axes[0,0].set_xlabel('scan count per edge')
    axes[0,0].set_ylabel('#edges')
    axes[0,0].grid(alpha=0.25)

    axes[0,1].hist(last_ages, bins=40, color='tab:orange', alpha=0.8)
    axes[0,1].axvline(AGE_SCALE_S, color='k', linestyle='--', linewidth=1, label='AGE_SCALE')
    axes[0,1].set_title('Final edge staleness distribution')
    axes[0,1].set_xlabel('final age [s]')
    axes[0,1].set_ylabel('#edges')
    axes[0,1].legend()
    axes[0,1].grid(alpha=0.25)

    axes[1,0].scatter(w_vals, visit_rate, s=8, alpha=0.35, color='tab:green')
    axes[1,0].set_title('Static weight vs scan rate')
    axes[1,0].set_xlabel('W(edge)')
    axes[1,0].set_ylabel('scan rate [scans/hour]')
    axes[1,0].grid(alpha=0.25)

    order = np.argsort(scan_counts)[::-1]
    scans_sorted = scan_counts[order]
    cum_scans = np.cumsum(scans_sorted) / max(scans_sorted.sum(), 1.0)
    cum_edges = np.arange(1, len(scans_sorted)+1) / len(scans_sorted)
    axes[1,1].plot(cum_edges, cum_scans, color='tab:red', linewidth=2)
    axes[1,1].plot([0,1], [0,1], '--', color='gray', linewidth=1)
    axes[1,1].set_title('Scan concentration (Pareto curve)')
    axes[1,1].set_xlabel('fraction of edges (highest scanned first)')
    axes[1,1].set_ylabel('fraction of all scans captured')
    axes[1,1].grid(alpha=0.25)

    plt.tight_layout()
    plt.show()
else:
    print('Skipping plots because matplotlib is not available.')

## 10) Spatial diagnostics

In [ ]:
if HAS_MPL:
    pos = {n: (UG.nodes[n]['x'], UG.nodes[n]['y']) for n in UG.nodes()}
    edgelist = list(UG.edges())
    edge_to_idx = {e: i for i, e in enumerate(sim['edges'])}

    scans_draw = np.array([visit_rate[edge_to_idx[e]] for e in edgelist], dtype=float)
    age_draw = np.array([SIM_HORIZON_S - sim['last_visit'][e] for e in edgelist], dtype=float)

    def draw_metric(ax, vals, title, cmap):
        vmin = float(vals.min())
        vmax = float(vals.max())
        if vmax <= vmin:
            vmax = vmin + 1e-9
        norm = mcolors.Normalize(vmin=vmin, vmax=vmax)
        colors = plt.get_cmap(cmap)(norm(vals))
        widths = 0.2 + 1.8 * np.clip((vals - vmin) / (vmax - vmin), 0, 1)
        nx.draw_networkx_edges(UG, pos, edgelist=edgelist, edge_color=colors, width=widths, ax=ax)
        sm = plt.cm.ScalarMappable(cmap=plt.get_cmap(cmap), norm=norm)
        sm.set_array([])
        plt.colorbar(sm, ax=ax, shrink=0.72, label=title)
        ax.set_title(title)
        ax.set_aspect('equal')
        ax.axis('off')

    fig, axes = plt.subplots(1, 2, figsize=(16, 7))
    draw_metric(axes[0], scans_draw, 'Edge scan rate [scans/hour]', 'viridis')
    draw_metric(axes[1], age_draw, 'Final edge age [s]', 'magma_r')
    plt.tight_layout()
    plt.show()
else:
    print('Skipping map plots because matplotlib is not available.')

## 11) Expected vs observed attention alignment

In [ ]:
edge_to_idx = {e: i for i, e in enumerate(sim['edges'])}
edgelist = list(UG.edges())
weight_draw = np.array([W[e] for e in edgelist], dtype=float)
scan_draw = np.array([visit_rate[edge_to_idx[e]] for e in edgelist], dtype=float)

w_cap = max(np.percentile(weight_draw, 95), 1e-9)
s_cap = max(np.percentile(scan_draw, 95), 1e-9)
weight_norm = np.clip(weight_draw / w_cap, 0, 1)
scan_norm = np.clip(scan_draw / s_cap, 0, 1)
delta = scan_norm - weight_norm

pearson = float(np.corrcoef(weight_draw, scan_draw)[0, 1])
print('Expected vs observed attention')
print('-----------------------------')
print(f'Pearson corr(weight, scan_rate): {pearson:.3f}')
print('{:<8} {:>28} {:>44}'.format('Top-k', 'Overlap(weight vs scan)', 'Share of scans captured by top-weight set'))
for k in [5, 10, 20]:
    n = max(1, int(np.ceil(len(edgelist) * (k / 100.0))))
    idx_w = np.argsort(weight_draw)[-n:]
    idx_s = np.argsort(scan_draw)[-n:]
    overlap = 100.0 * len(set(idx_w).intersection(set(idx_s))) / n
    share_scans = 100.0 * scan_draw[idx_w].sum() / max(scan_draw.sum(), 1e-9)
    print('{:>3}% {:>24.1f}% {:>36.1f}%'.format(k, overlap, share_scans))

under_idx = np.argsort(delta)[:10]
over_idx = np.argsort(delta)[-10:][::-1]

print('\nMost under-served edges:')
print('{:<35} {:>7} {:>9} {:>8}'.format('edge', 'W', 'rate/h', 'delta'))
for i in under_idx:
    u, v = edgelist[i]
    print('{:<15}-> {:<15} {:>7.3f} {:>9.2f} {:>8.3f}'.format(str(u)[:15], str(v)[:15], weight_draw[i], scan_draw[i], delta[i]))

print('\nMost over-served edges:')
print('{:<35} {:>7} {:>9} {:>8}'.format('edge', 'W', 'rate/h', 'delta'))
for i in over_idx:
    u, v = edgelist[i]
    print('{:<15}-> {:<15} {:>7.3f} {:>9.2f} {:>8.3f}'.format(str(u)[:15], str(v)[:15], weight_draw[i], scan_draw[i], delta[i]))

In [ ]:
if HAS_MPL:
    pos = {n: (UG.nodes[n]['x'], UG.nodes[n]['y']) for n in UG.nodes()}
    edgelist = list(UG.edges())

    d_lim = max(np.percentile(np.abs(delta), 95), 1e-6)

    def draw_metric(ax, vals, title, cmap, vmin, vmax, center=None):
        if center is None:
            norm = mcolors.Normalize(vmin=vmin, vmax=vmax)
        else:
            norm = mcolors.TwoSlopeNorm(vmin=vmin, vcenter=center, vmax=vmax)
        colors = plt.get_cmap(cmap)(norm(vals))
        widths = 0.2 + 1.8 * np.clip(np.abs(vals) / max(abs(vmin), abs(vmax), 1e-9), 0, 1)
        nx.draw_networkx_edges(UG, pos, edgelist=edgelist, edge_color=colors, width=widths, ax=ax)
        ax.scatter([sim['station_xy'][0]], [sim['station_xy'][1]], c='cyan', edgecolors='black', s=90, marker='*', zorder=5)
        sm = plt.cm.ScalarMappable(cmap=plt.get_cmap(cmap), norm=norm)
        sm.set_array([])
        plt.colorbar(sm, ax=ax, shrink=0.75)
        ax.set_title(title)
        ax.set_aspect('equal')
        ax.axis('off')

    fig, axes = plt.subplots(1, 3, figsize=(24, 7))
    draw_metric(axes[0], weight_norm, 'Expected attention (W norm)', 'plasma', 0.0, 1.0)
    draw_metric(axes[1], scan_norm, 'Observed attention (scan norm)', 'viridis', 0.0, 1.0)
    draw_metric(axes[2], delta, 'Alignment delta (observed - expected)', 'coolwarm', -d_lim, d_lim, center=0.0)
    plt.tight_layout()
    plt.show()
else:
    print('Skipping alignment maps because matplotlib is not available.')

## 12) GIF evolution (inline, optional)

In [ ]:
if not HAS_MPL:
    print('Skipping GIF generation because matplotlib is not available.')
elif not HAS_IMAGEIO:
    print('Skipping GIF generation because imageio is not available. Install with: pip install imageio')
else:
    GIF_SIM_HORIZON_S = 7200
    GIF_FRAME_EVERY_S = 120
    GIF_FPS = 6
    GIF_MAX_FRAMES = 120
    SAVE_GIF_TO_FILE = False
    GIF_PATH = 'aco_evolution.gif'

    # Reuse quick simulation snapshots from existing run data where possible
    # For a strict frame-by-frame replay, rerun a short simulation.
    print('Generating GIF frames...')

    # Build a lightweight replay by sampling synthetic interpolation between known states
    # (keeps cell compact and readable).
    frame_times = np.arange(GIF_FRAME_EVERY_S, min(GIF_SIM_HORIZON_S, SIM_HORIZON_S) + 1, GIF_FRAME_EVERY_S)
    frames = []

    edge_to_idx = {e: i for i, e in enumerate(sim['edges'])}
    edgelist = list(UG.edges())
    pos = {n: (UG.nodes[n]['x'], UG.nodes[n]['y']) for n in UG.nodes()}

    w_vals = np.array([W[e] for e in edgelist], dtype=float)
    w_cap = max(np.percentile(w_vals, 95), 1e-9)
    w_norm = np.clip(w_vals / w_cap, 0, 1)

    for t in frame_times[:GIF_MAX_FRAMES]:
        # approximate scan-rate/freshness from final counters as a compact illustrative replay
        scan_rate_t = (scan_counts / max((t / 3600.0), 1e-9))
        scan_draw_t = np.array([scan_rate_t[edge_to_idx[e]] for e in edgelist], dtype=float)
        s_cap = max(np.percentile(scan_draw_t, 95), 1e-9)
        s_norm = np.clip(scan_draw_t / s_cap, 0, 1)

        age_t = np.array([max(t - sim['last_visit'][e], 0.0) for e in edgelist], dtype=float)
        a_cap = max(np.percentile(age_t, 95), 1e-9)
        freshness_t = 1.0 - np.clip(age_t / a_cap, 0, 1)

        fig, axes = plt.subplots(1, 3, figsize=(18, 6))

        def draw(ax, vals, title, cmap):
            norm = mcolors.Normalize(vmin=0.0, vmax=1.0)
            colors = plt.get_cmap(cmap)(norm(vals))
            widths = 0.2 + 1.8 * vals
            nx.draw_networkx_edges(UG, pos, edgelist=edgelist, edge_color=colors, width=widths, ax=ax)
            ax.scatter([sim['station_xy'][0]], [sim['station_xy'][1]], c='cyan', edgecolors='black', s=80, marker='*', zorder=5)
            ax.set_title(title)
            ax.set_aspect('equal')
            ax.axis('off')

        draw(axes[0], w_norm, 'Expected attention (W norm)', 'plasma')
        draw(axes[1], s_norm, f'Observed scan rate (t={int(t)}s)', 'viridis')
        draw(axes[2], freshness_t, 'Freshness (higher=more recent)', 'magma')

        fig.suptitle(f'ACO evolution | t={int(t)}s', fontsize=12)
        fig.tight_layout()
        fig.canvas.draw()
        rgba = np.asarray(fig.canvas.buffer_rgba())
        frame = rgba[:, :, :3].copy()
        frames.append(frame)
        plt.close(fig)

    buf = BytesIO()
    imageio.mimsave(buf, frames, format='GIF', fps=GIF_FPS)
    buf.seek(0)

    if SAVE_GIF_TO_FILE:
        with open(GIF_PATH, 'wb') as f:
            f.write(buf.getvalue())
        print(f'GIF also saved to: {GIF_PATH}')

    if HAS_IPY:
        display(Image(data=buf.getvalue(), format='gif'))
    else:
        print('IPython display unavailable. Set SAVE_GIF_TO_FILE=True to save output.')

    print(f'GIF frames: {len(frames)}, FPS: {GIF_FPS}')